# RL Training Notebook (Google Colab)

Online RL against the unchanged Phase 04 rubric (`0.1.2`). This notebook:

1. builds train / held-out splits with the existing generator;
2. runs CPU-tested contracts before any GPU work;
3. scores only parseable completions (malformed output stays unscored);
4. trains with group-relative policy gradient + frozen-reference KL;
5. skips unscored or degenerate groups instead of crashing or faking rewards;
6. evaluates held-out tasks before and after training;
7. compares two KL coefficients and audits transcripts for reward gaming.

**Run cells in order.** Cell 1 defines `OUTPUT_ROOT` on Drive. Typical wall time with skip/curriculum: base held-out ~25–40 min, preflight ~5 min, smoke ~5 min, each 30-step beta run ~20–35 min.

Phase 04 rubric, eval harness, sandbox, and dashboard are not modified.

In [ ]:
import os
import subprocess
from pathlib import Path

assert os.path.exists('/content'), 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi'], check=True)

REPO_URL = 'https://github.com/prashere/evaluator_gym.git'
try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = os.environ.get('GITHUB_TOKEN')
if github_token:
    REPO_URL = f'https://{github_token}@github.com/prashere/evaluator_gym.git'

REPO = Path('/content/evaluator_gym')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run([
    'python', '-m', 'pip', 'install',
    'transformers==4.57.6', 'peft==0.17.1', 'bitsandbytes==0.47.0',
    'accelerate==1.10.1', 'mlflow==3.10.0', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5',
], check=True)

from google.colab import drive
drive.mount('/content/drive')

from evaluator_gym.training.phase07_core import DEFAULT_OUTPUT_ROOT, ensure_output_root

OUTPUT_ROOT = ensure_output_root(DEFAULT_OUTPUT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import json
from collections import Counter

from evaluator_gym.training.phase07_core import build_phase07_splits

TRAIN_CONFIG, HELDOUT_POOL_CONFIG, TRAIN_TASK_ROWS, HELDOUT_TASK_ROWS = build_phase07_splits()
TRAIN_TASKS = [row.as_dict() for row in TRAIN_TASK_ROWS]
HELDOUT_TASKS = [row.as_dict() for row in HELDOUT_TASK_ROWS]

print('Train:', TRAIN_CONFIG.to_dict())
print('Held-out pool:', HELDOUT_POOL_CONFIG.to_dict())
print('Tier counts:', dict(Counter(row['tier'] for row in TRAIN_TASKS)))
print('Held-out IDs:', [row['task_id'] for row in HELDOUT_TASKS])

In [ ]:
import asyncio
import concurrent.futures
import random
from statistics import fmean, stdev

import mlflow
import numpy as np
import torch
from evaluator_gym.parser import parse_agent_response
from evaluator_gym.rubric import RUBRIC_VERSION, score_task
from evaluator_gym.rubric.audit import breakdown_to_dict
from evaluator_gym.training.phase07_core import (
    BETAS,
    CURRICULUM_TIER1_STEPS,
    DEFAULT_OUTPUT_ROOT,
    GROUP_SIZE,
    HELDOUT_ROLLOUTS,
    MAX_COMPLETION_TOKENS,
    MAX_RETRIES,
    TRAIN_STEPS,
    ensure_output_root,
    is_degenerate_group,
    select_training_task,
    summarize_evaluation,
    summarize_preflight,
    summarize_rejections,
)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
RUN_SEED = 20260913

OUTPUT_ROOT = ensure_output_root(globals().get('OUTPUT_ROOT', DEFAULT_OUTPUT_ROOT))


def run_async(coroutine):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coroutine)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def append_jsonl(path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, sort_keys=True, default=str) + '\n')
        handle.flush()
        os.fsync(handle.fileno())


def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


async def score_text(task, text):
    parsed = parse_agent_response(text, task['info'])
    if not parsed.ok:
        return None, {
            'ok': False,
            'error_class': parsed.error_class,
            'error_message': parsed.error_message,
        }, None
    breakdown = await score_task(
        parsed=parsed.data or {},
        ground_truth=task['ground_truth'],
        info=task['info'],
        mode='single',
        completion=[{'role': 'assistant', 'content': text}],
        parse_result={'ok': True, 'data': parsed.data},
    )
    return breakdown.final_reward, {'ok': True, 'data': parsed.data}, breakdown_to_dict(breakdown)


assert RUBRIC_VERSION == '0.1.2', f'Expected the evaluation rubric, got {RUBRIC_VERSION}'
assert torch.cuda.is_available(), 'A Colab GPU is required'
print('GPU:', torch.cuda.get_device_name(0))
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('MAX_COMPLETION_TOKENS:', MAX_COMPLETION_TOKENS, '| MAX_RETRIES:', MAX_RETRIES)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed


def build_policy():
    set_seed(RUN_SEED)
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        quantization_config=quantization,
        device_map={'': 0},
        torch_dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ))
    unexpected = [name for name, parameter in model.named_parameters() if parameter.requires_grad and 'lora_' not in name]
    assert not unexpected, f'Frozen-reference invariant failed: {unexpected[:5]}'
    return model, tokenizer


def render_prompt(task, tokenizer):
    text = tokenizer.apply_chat_template(task['prompt'], tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(text, return_tensors='pt').to('cuda')
    assert encoded['input_ids'].shape[1] + MAX_COMPLETION_TOKENS <= 32768
    return encoded


def generate_valid_group(model, tokenizer, task, step, rejection_path):
    encoded = render_prompt(task, tokenizer)
    prompt_length = encoded['input_ids'].shape[1]
    samples = []
    model.eval()
    for group_index in range(GROUP_SIZE):
        accepted = None
        for retry in range(MAX_RETRIES):
            sample_seed = RUN_SEED + step * 1000 + group_index * 10 + retry
            torch.manual_seed(sample_seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded,
                    do_sample=True,
                    temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            completion_ids = output[0, prompt_length:].detach()
            text = tokenizer.decode(completion_ids, skip_special_tokens=True)
            reward, parse_result, audit = run_async(score_text(task, text))
            candidate = {
                'task_id': task['task_id'], 'tier': task['tier'], 'step': step,
                'group_index': group_index, 'retry': retry, 'seed': sample_seed,
                'completion': text, 'completion_tokens': int(completion_ids.numel()),
                'reward': reward, 'parse_result': parse_result, 'reward_audit': audit,
            }
            if reward is None:
                append_jsonl(rejection_path, candidate)
                continue
            candidate['prompt_ids'] = encoded['input_ids'][0].detach()
            candidate['completion_ids'] = completion_ids
            accepted = candidate
            break
        if accepted is None:
            model.train()
            return None
        samples.append(accepted)
    model.train()
    return samples


def token_statistics(model, sample):
    tokens = torch.cat([sample['prompt_ids'], sample['completion_ids']]).unsqueeze(0)
    targets = sample['completion_ids']
    keep = targets.numel() + 1
    policy_logits = model(tokens, logits_to_keep=keep).logits[0, :-1].float()
    policy_log_probs = policy_logits.log_softmax(-1)
    policy_token_logp = policy_log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    entropy = -(policy_log_probs.exp() * policy_log_probs).sum(-1).mean()
    with torch.no_grad(), model.disable_adapter():
        reference_logits = model(tokens, logits_to_keep=keep).logits[0, :-1].float()
        reference_token_logp = reference_logits.log_softmax(-1).gather(1, targets.unsqueeze(1)).squeeze(1)
    log_ratio = reference_token_logp - policy_token_logp
    k3 = (torch.exp(log_ratio) - log_ratio - 1).mean()
    return policy_token_logp.mean(), k3, entropy

In [ ]:
from tqdm.auto import tqdm


def evaluate_policy(model, tokenizer, run_name):
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    transcript_path = run_dir / 'heldout_rollouts.jsonl'
    if transcript_path.exists():
        transcript_path.unlink()
    model.eval()
    total = len(HELDOUT_TASKS) * HELDOUT_ROLLOUTS
    progress = tqdm(total=total, desc=f'held-out {run_name}')
    for task_index, task in enumerate(HELDOUT_TASKS):
        encoded = render_prompt(task, tokenizer)
        prompt_length = encoded['input_ids'].shape[1]
        for rollout_index in range(HELDOUT_ROLLOUTS):
            seed = RUN_SEED + task_index * HELDOUT_ROLLOUTS + rollout_index
            torch.manual_seed(seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded, do_sample=True, temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            ids = output[0, prompt_length:]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            reward, parse_result, audit = run_async(score_text(task, text))
            append_jsonl(transcript_path, {
                'run': run_name, 'task_id': task['task_id'], 'tier': task['tier'],
                'rollout_index': rollout_index, 'seed': seed, 'completion': text,
                'completion_tokens': int(ids.numel()), 'reward': reward,
                'parse_result': parse_result, 'reward_audit': audit,
            })
            progress.update(1)
            progress.set_postfix(task=task['task_id'], tier=task['tier'], scored=reward is not None)
    progress.close()
    rows = read_jsonl(transcript_path)
    summary = summarize_evaluation(rows)
    (run_dir / 'heldout_summary.json').write_text(json.dumps(summary, indent=2))
    return summary


base_model, base_tokenizer = build_policy()
BASE_HELDOUT = evaluate_policy(base_model, base_tokenizer, 'base')
print(json.dumps(BASE_HELDOUT, indent=2))
del base_model
torch.cuda.empty_cache()

In [ ]:
from peft import get_peft_model_state_dict, set_peft_model_state_dict


def save_checkpoint(run_dir, step, model, optimizer):
    checkpoint = run_dir / f'checkpoint-{step}'
    checkpoint.mkdir(parents=True, exist_ok=True)
    torch.save({
        'step': step,
        'adapter': get_peft_model_state_dict(model),
        'optimizer': optimizer.state_dict(),
        'python_rng': random.getstate(),
        'numpy_rng': np.random.get_state(),
        'torch_rng': torch.get_rng_state(),
        'cuda_rng': torch.cuda.get_rng_state_all(),
    }, checkpoint / 'state.pt')
    return checkpoint


def run_preflight(model, tokenizer):
    probes = []
    model.eval()
    for task_index, task_row in enumerate(TRAIN_TASK_ROWS):
        task = task_row.as_dict()
        encoded = render_prompt(task, tokenizer)
        prompt_length = encoded['input_ids'].shape[1]
        accepted = None
        last_probe = None
        for retry in range(MAX_RETRIES):
            seed = RUN_SEED + 500000 + task_index * 10 + retry
            torch.manual_seed(seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded, do_sample=True, temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            ids = output[0, prompt_length:]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            reward, parse_result, _ = run_async(score_text(task, text))
            last_probe = {
                'task_id': task['task_id'], 'tier': task['tier'], 'retry': retry,
                'reward': reward, 'parse_result': parse_result,
                'completion_tokens': int(ids.numel()),
            }
            if reward is not None:
                accepted = last_probe
                break
        probes.append(accepted or last_probe)
    model.train()
    return summarize_preflight(probes)


def train_run(beta, run_name, total_steps, trainable_ids, resume_checkpoint=None):
    assert beta > 0
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / 'metrics.jsonl'
    rollout_path = run_dir / 'training_rollouts.jsonl'
    rejection_path = run_dir / 'rejected_unscored.jsonl'
    config_path = run_dir / 'config.json'
    run_config = {
        'model': MODEL_ID, 'model_revision': MODEL_REVISION, 'rubric_version': RUBRIC_VERSION,
        'beta': beta, 'seed': RUN_SEED, 'total_steps': total_steps, 'group_size': GROUP_SIZE,
        'max_retries': MAX_RETRIES, 'max_completion_tokens': MAX_COMPLETION_TOKENS,
        'curriculum_tier1_steps': CURRICULUM_TIER1_STEPS,
        'trainable_task_ids': sorted(trainable_ids),
        'train_generator': TRAIN_CONFIG.to_dict(),
        'heldout_pool_generator': HELDOUT_POOL_CONFIG.to_dict(),
        'train_task_ids': [row['task_id'] for row in TRAIN_TASKS],
        'heldout_task_ids': [row['task_id'] for row in HELDOUT_TASKS],
    }
    model, tokenizer = build_policy()
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-5)
    start_step = 0
    if resume_checkpoint:
        state = torch.load(Path(resume_checkpoint) / 'state.pt', map_location='cpu', weights_only=False)
        set_peft_model_state_dict(model, state['adapter'])
        optimizer.load_state_dict(state['optimizer'])
        random.setstate(state['python_rng'])
        np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        start_step = state['step']
    elif metrics_path.exists() or rollout_path.exists():
        raise RuntimeError(f'{run_dir} already contains a run; resume it or use another run name')
    config_path.write_text(json.dumps(run_config, indent=2))

    mlflow.set_tracking_uri((OUTPUT_ROOT / 'mlruns').as_uri())
    mlflow.set_experiment('phase07-online-rl')
    trained_steps = 0
    skipped_unscored = 0
    skipped_degenerate = 0
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'model': MODEL_ID, 'model_revision': MODEL_REVISION,
            'rubric_version': RUBRIC_VERSION, 'beta': beta,
            'train_seed': TRAIN_CONFIG.seed, 'heldout_pool_seed': HELDOUT_POOL_CONFIG.seed,
            'group_size': GROUP_SIZE, 'max_retries': MAX_RETRIES,
            'trainable_tasks': len(trainable_ids),
        })
        progress = tqdm(range(start_step, total_steps), desc=run_name, initial=start_step, total=total_steps)
        for step in progress:
            task_row = select_training_task(step, TRAIN_TASK_ROWS, trainable_ids)
            if task_row is None:
                metric = {
                    'step': step + 1, 'task_id': None, 'tier': None,
                    'optimizer_applied': False, 'skip_reason': 'no_trainable_task',
                }
                append_jsonl(metrics_path, metric)
                progress.set_postfix(step=step + 1, skip='no_task')
                continue
            task = task_row.as_dict()
            samples = generate_valid_group(model, tokenizer, task, step, rejection_path)
            if samples is None:
                skipped_unscored += 1
                metric = {
                    'step': step + 1, 'task_id': task['task_id'], 'tier': task['tier'],
                    'optimizer_applied': False, 'skip_reason': 'skipped_unscored_group',
                    'rejection_summary': summarize_rejections(
                        [row for row in read_jsonl(rejection_path) if row.get('step') == step]
                    ),
                }
                append_jsonl(metrics_path, metric)
                progress.set_postfix(step=step + 1, task=task['task_id'], skip='unscored')
                continue
            reward_values = [float(sample['reward']) for sample in samples]
            if is_degenerate_group(reward_values):
                skipped_degenerate += 1
                metric = {
                    'step': step + 1, 'task_id': task['task_id'], 'tier': task['tier'],
                    'mean_reward': fmean(reward_values), 'reward_std': 0.0,
                    'degenerate_group': 1.0, 'optimizer_applied': False,
                    'skip_reason': 'skipped_degenerate_group',
                    'exact_pass_rate': sum(value == 1.0 for value in reward_values) / GROUP_SIZE,
                    'mean_completion_length': fmean(sample['completion_tokens'] for sample in samples),
                }
                append_jsonl(metrics_path, metric)
                for sample in samples:
                    append_jsonl(rollout_path, {k: v for k, v in sample.items() if k not in ('prompt_ids', 'completion_ids')})
                progress.set_postfix(step=step + 1, task=task['task_id'], skip='degenerate')
                continue

            rewards = torch.tensor(reward_values, device='cuda')
            reward_std = rewards.std()
            advantages = (rewards - rewards.mean()) / (reward_std + 1e-4)
            optimizer.zero_grad()
            policy_logps, kls, entropies = [], [], []
            for sample in samples:
                policy_logp, k3, entropy = token_statistics(model, sample)
                policy_logps.append(policy_logp)
                kls.append(k3)
                entropies.append(entropy)
            policy_logps = torch.stack(policy_logps)
            kls = torch.stack(kls)
            loss = -(advantages.detach() * policy_logps).mean() + beta * kls.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            trained_steps += 1

            metric = {
                'step': step + 1, 'task_id': task['task_id'], 'tier': task['tier'],
                'mean_reward': rewards.mean().item(), 'reward_std': reward_std.item(),
                'kl': kls.mean().item(), 'entropy': torch.stack(entropies).mean().item(),
                'mean_completion_length': fmean(sample['completion_tokens'] for sample in samples),
                'exact_pass_rate': sum(sample['reward'] == 1.0 for sample in samples) / GROUP_SIZE,
                'tier_1_pass_rate': sum(sample['reward'] == 1.0 for sample in samples) / GROUP_SIZE if task['tier'] == 1 else None,
                'tier_2_pass_rate': sum(sample['reward'] == 1.0 for sample in samples) / GROUP_SIZE if task['tier'] == 2 else None,
                'tier_3_pass_rate': sum(sample['reward'] == 1.0 for sample in samples) / GROUP_SIZE if task['tier'] == 3 else None,
                'degenerate_group': 0.0, 'optimizer_applied': True, 'skip_reason': None,
                'loss': loss.item(),
            }
            append_jsonl(metrics_path, metric)
            mlflow.log_metrics({k: v for k, v in metric.items() if isinstance(v, (int, float))}, step=step + 1)
            for sample in samples:
                append_jsonl(rollout_path, {k: v for k, v in sample.items() if k not in ('prompt_ids', 'completion_ids')})
            progress.set_postfix(step=step + 1, task=task['task_id'], reward=f'{metric["mean_reward"]:.2f}', trained=trained_steps)
            if (step + 1) % 5 == 0 or step + 1 == total_steps:
                save_checkpoint(run_dir, step + 1, model, optimizer)
        progress.close()
        print(f'{run_name}: trained={trained_steps}, skipped_unscored={skipped_unscored}, skipped_degenerate={skipped_degenerate}')
    model.save_pretrained(run_dir / 'final-adapter')
    tokenizer.save_pretrained(run_dir / 'final-adapter')
    return model, tokenizer, run_dir


preflight_model, preflight_tokenizer = build_policy()
PREFLIGHT = run_preflight(preflight_model, preflight_tokenizer)
TRAINABLE_IDS = set(PREFLIGHT['trainable_task_ids'])
(OUTPUT_ROOT / 'preflight.json').write_text(json.dumps(PREFLIGHT, indent=2))
print('Trainable tasks:', PREFLIGHT['trainable_by_tier'])
del preflight_model
torch.cuda.empty_cache()

smoke_model, smoke_tokenizer, smoke_dir = train_run(1e-3, 'smoke', 2, TRAINABLE_IDS)
del smoke_model
torch.cuda.empty_cache()
smoke_model, smoke_tokenizer, smoke_dir = train_run(1e-3, 'smoke', 3, TRAINABLE_IDS, smoke_dir / 'checkpoint-2')
smoke_metrics = read_jsonl(smoke_dir / 'metrics.jsonl')
assert len(smoke_metrics) == 3
assert {row['step'] for row in smoke_metrics} == {1, 2, 3}
assert sum(row.get('optimizer_applied') for row in smoke_metrics) >= 1
del smoke_model
torch.cuda.empty_cache()
print('Smoke and resume passed')

In [ ]:
low_model, low_tokenizer, low_dir = train_run(1e-5, 'beta-1e-5', TRAIN_STEPS, TRAINABLE_IDS)
LOW_HELDOUT = evaluate_policy(low_model, low_tokenizer, 'beta-1e-5')
del low_model
torch.cuda.empty_cache()

controlled_model, controlled_tokenizer, controlled_dir = train_run(1e-3, 'beta-1e-3', TRAIN_STEPS, TRAINABLE_IDS)
CONTROLLED_HELDOUT = evaluate_policy(controlled_model, controlled_tokenizer, 'beta-1e-3')
del controlled_model
torch.cuda.empty_cache()

COMPARISON = {'base': BASE_HELDOUT, 'beta-1e-5': LOW_HELDOUT, 'beta-1e-3': CONTROLLED_HELDOUT}
(OUTPUT_ROOT / 'heldout_comparison.json').write_text(json.dumps(COMPARISON, indent=2))
print(json.dumps(COMPARISON, indent=2))

In [ ]:
import matplotlib.pyplot as plt

FIGURE_DIR = OUTPUT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)
runs = {
    'beta=1e-5': read_jsonl(low_dir / 'metrics.jsonl'),
    'beta=1e-3': read_jsonl(controlled_dir / 'metrics.jsonl'),
}


def save_curve(field, ylabel, filename):
    figure, axis = plt.subplots(figsize=(7, 4))
    for label, rows in runs.items():
        points = [(row['step'], row[field]) for row in rows if row.get(field) is not None and row.get('optimizer_applied')]
        axis.plot([x for x, _ in points], [y for _, y in points], marker='o', markersize=3, label=label)
    axis.set(xlabel='Optimizer step', ylabel=ylabel, title=ylabel)
    axis.grid(alpha=0.25)
    axis.legend()
    figure.savefig(FIGURE_DIR / f'{filename}.png', bbox_inches='tight')
    plt.show()


save_curve('mean_reward', 'Mean Phase 04 reward', 'reward')
save_curve('kl', 'Frozen-reference k3 KL', 'kl')
save_curve('entropy', 'Policy token entropy', 'entropy')
save_curve('mean_completion_length', 'Mean completion length', 'completion-length')

figure, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for tier, axis in zip((1, 2, 3), axes):
    field = f'tier_{tier}_pass_rate'
    for label, rows in runs.items():
        points = [(row['step'], row[field]) for row in rows if row.get(field) is not None and row.get('optimizer_applied')]
        axis.plot([x for x, _ in points], [y for _, y in points], marker='o', label=label)
    axis.set(title=f'Tier {tier}', xlabel='Optimizer step')
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Exact pass rate')
axes[-1].legend()
figure.tight_layout()
figure.savefig(FIGURE_DIR / 'per-tier-pass-rate.png', bbox_inches='tight')
plt.show()

labels = list(COMPARISON)
means = [COMPARISON[label]['mean_reward_scored'] or 0 for label in labels]
figure, axis = plt.subplots(figsize=(7, 4))
axis.bar(labels, means)
axis.set(title='Held-out before and after', ylabel='Mean scored reward', ylim=(0, 1))
figure.savefig(FIGURE_DIR / 'heldout-before-after.png', bbox_inches='tight')
plt.show()


def strict_json(text):
    try:
        return isinstance(json.loads(text.strip()), dict)
    except json.JSONDecodeError:
        return False


def exploit_search(run_dir):
    rows = read_jsonl(run_dir / 'training_rollouts.jsonl')
    outputs = [' '.join(row['completion'].split()) for row in rows]
    modal = Counter(outputs).most_common(1)[0] if outputs else (None, 0)
    rewards = np.array([row['reward'] for row in rows], dtype=float)
    lengths = np.array([row['completion_tokens'] for row in rows], dtype=float)
    correlation = float(np.corrcoef(rewards, lengths)[0, 1]) if len(rows) > 1 and rewards.std() and lengths.std() else None
    metrics_rows = read_jsonl(run_dir / 'metrics.jsonl')
    candidates = {
        'longest': sorted(rows, key=lambda row: row['completion_tokens'], reverse=True)[:10],
        'highest_reward': sorted(rows, key=lambda row: row['reward'], reverse=True)[:10],
        'strict_json_disagreements': [row for row in rows if row['parse_result']['ok'] and not strict_json(row['completion'])][:10],
    }
    return {
        'status': 'requires_manual_transcript_review',
        'modal_output_fraction': modal[1] / len(rows) if rows else None,
        'reward_length_correlation': correlation,
        'unscored_rejections': len(read_jsonl(run_dir / 'rejected_unscored.jsonl')),
        'degenerate_group_fraction': fmean(row.get('degenerate_group', 0.0) for row in metrics_rows) if metrics_rows else None,
        'skipped_fraction': fmean(not row.get('optimizer_applied', False) for row in metrics_rows) if metrics_rows else None,
        'candidates': candidates,
        'checks': ['entropy collapse', 'KL blow-up', 'length hacking', 'format collapse', 'degenerate groups'],
        'finding': None,
    }


AUDIT = {'beta-1e-5': exploit_search(low_dir), 'beta-1e-3': exploit_search(controlled_dir)}
(OUTPUT_ROOT / 'exploit_search.json').write_text(json.dumps(AUDIT, indent=2))
with mlflow.start_run(run_name='phase07-report'):
    mlflow.log_artifacts(str(FIGURE_DIR), artifact_path='figures')
    mlflow.log_artifact(str(OUTPUT_ROOT / 'heldout_comparison.json'))
    mlflow.log_artifact(str(OUTPUT_ROOT / 'exploit_search.json'))
    mlflow.log_artifact(str(OUTPUT_ROOT / 'preflight.json'))
print(json.dumps(AUDIT, indent=2))
print('Manually review candidate transcripts before writing conclusions.')